# **scNET: Learning Context-Specific Gene and Cell Embeddings by Integrating Single-Cell Gene Expression Data with Protein-Protein Interaction Information**

### In this notebook, we demonstrate how to apply scNET to larger datasets.  
To reduce training time, we subsample cells during training and then apply the trained model to the remaining cells.


**Install scnet using pip**

**It may be necessary to restart (not delete) the runtime after installing scNET in Google Colab. We are working on fixing this issue in future releases.**

This is still a beta version, so please install scNET from TestPyPI.
---



In [ ]:
#!pip install scnet

! pip install --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple  scnet==0.2.4.2

!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

### For subsampled training, pass `subsample_size` to the `run_scNET` function.  
A subsample of 25,000 cells 


**Tablasapians data**

*we are working on TabulaSapiens with 180 cell type (cell_types.txt), contains 481,120 cells and 58,870 genes.The expression matrix values range from 0 to 10, which indicates the data has already been log-normalized (log1p transformed counts)*

In [ ]:
import scanpy as sc
import scNET
scNET.main.DE_GENES_NUM = 10000

obj = sc.read_h5ad("/workspace/sivakami/scNET/Data/TabulaSapiens.h5ad")
print('starting',flush=True)

In [ ]:

# gene names already present
obj.var_names_make_unique()

scNET.run_scNET(
    obj,
    pre_processing_flag=True,
    human_flag=True,
    number_of_batches=10,
    split_cells=True,
    save_model_flag=True,
    max_epoch=400,
    model_name="model",
    subsample_size=25000
)

### To apply the trained model to the entire object, use `scNET.apply_on_full_object`.  
The `batch_size` should be set to `number_of_cells // number_of_batches`, consistent with the training setup.


In [ ]:
recon_obj = scNET.apply_on_full_object(obj, "model", human_flag=True, batch_size=2500)

In [ ]:
sc.tl.umap(recon_obj,min_dist=0.4)
sc.pl.umap(recon_obj, color=["cell_type"],color_map="tab20",layer="embeddings")
sc.pl.umap(recon_obj, color=["CD14","CD19","MKI67","CD8A","SOX2","EPCAM","P2RY12","MAP2"])